# Sales Data Analysis with Basic Python

Analysis of the Superstore sales dataset using only built-in Python (csv module, lists, dictionaries, functions, loops).

Dataset: https://www.kaggle.com/datasets/vivek468/superstore-dataset-final

In [1]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/vivek468/superstore-dataset-final/Sample - Superstore.csv


# load the data

In [2]:
import csv

file_path = "/kaggle/input/datasets/vivek468/superstore-dataset-final/Sample - Superstore.csv"

orders = []
with open(file_path, encoding="latin-1") as f:
    reader = csv.DictReader(f)
    for row in reader:
        orders.append(row)

print("Total rows:", len(orders))
print("Columns:", list(orders[0].keys()))
print(orders[0])

Total rows: 9994
Columns: ['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit']
{'Row ID': '1', 'Order ID': 'CA-2016-152156', 'Order Date': '11/8/2016', 'Ship Date': '11/11/2016', 'Ship Mode': 'Second Class', 'Customer ID': 'CG-12520', 'Customer Name': 'Claire Gute', 'Segment': 'Consumer', 'Country': 'United States', 'City': 'Henderson', 'State': 'Kentucky', 'Postal Code': '42420', 'Region': 'South', 'Product ID': 'FUR-BO-10001798', 'Category': 'Furniture', 'Sub-Category': 'Bookcases', 'Product Name': 'Bush Somerset Collection Bookcase', 'Sales': '261.96', 'Quantity': '2', 'Discount': '0', 'Profit': '41.9136'}


# clean the data and convert text to numbers

In [3]:
clean_orders = []
skipped = 0

for row in orders:
    try:
        row["Sales"] = float(row["Sales"])
        row["Quantity"] = int(row["Quantity"])
        row["Discount"] = float(row["Discount"])
        row["Profit"] = float(row["Profit"])
        row["Region"] = row["Region"].strip()
        row["Category"] = row["Category"].strip()
        clean_orders.append(row)
    except (ValueError, KeyError):
        skipped += 1

print("Clean rows:", len(clean_orders))
print("Skipped rows:", skipped)

Clean rows: 9994
Skipped rows: 0


# total sales and profit by region

In [4]:
def total_by(column, value_column):
    totals = {}
    for row in clean_orders:
        key = row[column]
        totals[key] = totals.get(key, 0) + row[value_column]
    return totals

total_sales = sum(row["Sales"] for row in clean_orders)
total_profit = sum(row["Profit"] for row in clean_orders)
print(f"Total Sales: {total_sales:,.2f}")
print(f"Total Profit: {total_profit:,.2f}")

print("\nProfit by Region")
for region, profit in total_by("Region", "Profit").items():
    print(f"{region}: {profit:,.2f}")

Total Sales: 2,297,200.86
Total Profit: 286,397.02

Profit by Region
South: 46,749.43
West: 108,418.45
Central: 39,706.36
East: 91,522.78


# top 5 products by sales

In [5]:
def sort_by_value(data):
    return sorted(data.items(), key=lambda item: item[1], reverse=True)

product_sales = total_by("Product Name", "Sales")

print("Top 5 Products by Sales")
for name, sales in sort_by_value(product_sales)[:5]:
    print(f"{name}: {sales:,.2f}")

Top 5 Products by Sales
Canon imageCLASS 2200 Advanced Copier: 61,599.82
Fellowes PB500 Electric Punch Plastic Comb Binding Machine with Manual Bind: 27,453.38
Cisco TelePresence System EX90 Videoconferencing Unit: 22,638.48
HON 5400 Series Task Chairs for Big and Tall: 21,870.58
GBC DocuBind TL300 Electric Binding System: 19,823.48


# profit by category

In [6]:
print("Profit by Category")
for category, profit in total_by("Category", "Profit").items():
    print(f"{category}: {profit:,.2f}")

Profit by Category
Furniture: 18,451.27
Office Supplies: 122,490.80
Technology: 145,454.95


# Does a higher discount reduce profit?

In [7]:
def discount_band(discount):
    if discount == 0:
        return "No discount"
    elif discount <= 0.2:
        return "1-20%"
    elif discount <= 0.4:
        return "21-40%"
    else:
        return "Above 40%"

band_profit = {}
band_count = {}

for row in clean_orders:
    band = discount_band(row["Discount"])
    band_profit[band] = band_profit.get(band, 0) + row["Profit"]
    band_count[band] = band_count.get(band, 0) + 1

print("Average profit per order line by discount band")
for band in ["No discount", "1-20%", "21-40%", "Above 40%"]:
    average = band_profit[band] / band_count[band]
    print(f"{band}: {average:,.2f} ({band_count[band]} order lines)")

Average profit per order line by discount band
No discount: 66.90 (4798 order lines)
1-20%: 26.50 (3803 order lines)
21-40%: -77.86 (460 order lines)
Above 40%: -106.71 (933 order lines)


# which sub-categories lose money?

In [8]:
sub_profit = total_by("Sub-Category", "Profit")

print("Profit by Sub-Category (lowest first)")
for name, profit in sorted(sub_profit.items(), key=lambda item: item[1]):
    print(f"{name}: {profit:,.2f}")

Profit by Sub-Category (lowest first)
Tables: -17,725.48
Bookcases: -3,472.56
Supplies: -1,189.10
Fasteners: 949.52
Machines: 3,384.76
Labels: 5,546.25
Art: 6,527.79
Envelopes: 6,964.18
Furnishings: 13,059.14
Appliances: 18,138.01
Storage: 21,278.83
Chairs: 26,590.17
Binders: 30,221.76
Paper: 34,053.57
Accessories: 41,936.64
Phones: 44,515.73
Copiers: 55,617.82


# sales by year

In [9]:
year_sales = {}
for row in clean_orders:
    year = row["Order Date"].split("/")[2]
    year_sales[year] = year_sales.get(year, 0) + row["Sales"]

print("Sales by Year")
for year in sorted(year_sales):
    print(f"{year}: {year_sales[year]:,.2f}")

Sales by Year
2014: 484,247.50
2015: 470,532.51
2016: 609,205.60
2017: 733,215.26


# Key Insights

1. The business made a profit of 286,397 on sales of 2,297,201, a margin of about 12.5%.
2. West (108,418) and East (91,523) generate most of the profit. Central (39,706) is the weakest region.
3. Technology (145,455) and Office Supplies (122,491) drive profit. Furniture earns only 18,451.
4. Furniture's weak result comes from Tables (-17,725) and Bookcases (-3,473), which both lose money. Supplies also loses money (-1,189).
5. Discounts above 20% turn average profit per order line negative: -77.86 for 21-40% and -106.71 above 40%, compared with +66.90 with no discount.
6. Sales fell about 3% in 2015, then grew strongly to 733,215 in 2017.

# Recommendations
1. Cap discounts at 20%, since heavy discounting is linked to losses.
2. Review pricing and discounts on Tables and Bookcases.
3. Look into why Central earns the least profit.
4. Keep investing in Copiers, Phones and Accessories, which are the strongest profit makers.

# Limitations
Each row in the dataset is an order line, not a full order. The data shows that heavy discounts and losses go together, but it does not prove that discounts alone cause the losses.